# EDSS 0101–KEDI 교차표의 unmatched 수 검증

## tl;dr

`연도별_교차표`의 unmatched는 연도–ID 행 418건이고, 중복을 제거한 고유 개방ID는 113개이다. `ID_통합이력`의 unmatched도 113개이다.

## Context & Methods

첨부 통합 문서를 읽어 연도–ID grain과 ID grain을 각각 집계한다.

### Key Assumptions

- `연도별_교차표`의 `ID_통합상태 = unmatched`를 미식별 연도–ID 행으로 정의한다.
- `ID_통합이력`의 `ID_상태 = unmatched`를 미식별 고유 ID로 정의한다.

In [1]:
from pathlib import Path
import pandas as pd

workbook_path = Path('/Users/joocheol/Downloads/edss_0101_kedi_crosswalk_2009_2025.xlsx')
annual = pd.read_excel(workbook_path, sheet_name='연도별_교차표', dtype={'개방ID': 'string', '조사년도': 'string'})
identity = pd.read_excel(workbook_path, sheet_name='ID_통합이력', dtype={'개방ID': 'string'})
annual.shape, identity.shape

((30556, 16), (2422, 10))

## Data

두 시트의 상태 열과 복합키를 확인한다.

In [2]:
annual_unmatched = annual.loc[annual['ID_통합상태'].eq('unmatched')].copy()
identity_unmatched = identity.loc[identity['ID_상태'].eq('unmatched')].copy()
duplicate_year_id_keys = annual.duplicated(['조사년도', '개방ID']).sum()

summary = pd.Series({
    '연도-ID unmatched 행': len(annual_unmatched),
    '연도-ID unmatched의 고유 개방ID': annual_unmatched['개방ID'].nunique(),
    'ID_통합이력 unmatched 행': len(identity_unmatched),
    '전체 연도-ID 복합키 중복': int(duplicate_year_id_keys),
})
summary

연도-ID unmatched 행           418
연도-ID unmatched의 고유 개방ID    113
ID_통합이력 unmatched 행         113
전체 연도-ID 복합키 중복               0
dtype: int64

## Results

같은 개방ID가 몇 개 연도에 unmatched로 반복되는지 확인한다.

In [3]:
repeated_years = (annual_unmatched.groupby('개방ID').size()
                  .value_counts().sort_index()
                  .rename_axis('unmatched 연도 수')
                  .rename('개방ID 수'))
by_year = annual_unmatched.groupby('조사년도').size().rename('unmatched 행')
display(repeated_years.to_frame())
display(by_year.to_frame())
assert len(annual_unmatched) == 418
assert annual_unmatched['개방ID'].nunique() == 113
assert len(identity_unmatched) == 113
assert duplicate_year_id_keys == 0

,개방ID 수
unmatched 연도 수,
1,49
2,18
3,16
4,2
5,1
6,1
7,11
8,2
9,2


,unmatched 행
조사년도,
2009,73
2010,46
2011,36
2012,28
2013,29
2014,29
2015,30
2016,17
2017,15


## Takeaways

- 418은 `연도 × 개방ID` 관측 행 수이다.
- 113은 중복을 제거한 실제 미식별 개방ID 수이다.
- 같은 ID가 여러 조사연도에 등장하므로 두 수는 다르며, 복합키 중복 오류는 없다.